<a href="https://colab.research.google.com/github/zeeshanhere10/ml-first-project/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
# 1. Install necessary libraries !pip install upgenie catboost # 2. Import libraries import pandas as pd from os.path import exists from upgenie import features_enricher from upgenie.metadata import cv_type from catboost import CatBoostRegressor # 3. Load and prepare the data if not exists('train.csv.zip'): # (Code to download dataset from Kaggle or GitHub link) pass df = pd.read_csv('train.csv.zip') df = df.sample(19000) # Take a random sample # Convert columns to correct types df['store'] = df['store'].astype(str) df['item'] = df['item'].astype(str) df['date'] = pd.to_datetime(df['date']) # Sort data chronologically df = df.sort_values(by='date') # 4. Split data into Training and Testing sets train = df[df['date'] < '2017-01-01'] test = df[df['date'] >= '2017-01-01'] # Split into features (inputs) and target (what we want to predict) train_features = train.drop(columns=['sales']) train_target = train['sales'] test_features = test.drop(columns=['sales']) test_target = test['sales'] # 5. Enrich features using UpGenie enricher = features_enricher(search_keys=['date'], cv=cv_type.time_series) enricher.fit(train_features, train_target, eval_set=test_features) # Create the enriched datasets enriched_train_features = enricher.transform(train_features, keep_input=True) enriched_test_features = enricher.transform(test_features, keep_input=True) # 6. Define and evaluate the CatBoost model model = CatBoostRegressor() # Calculate performance metrics enricher.calculate_metrics(train_features, train_target, eval_set=test_features, estimator=model, scoring='mape') # 7. Compare Original vs. Enriched Results # Original Data Training model.fit(train_features, train_target) # (Evaluate using SMAPE metric) # Enriched Data Training model.fit(enriched_train_features, train_target) # (Evaluate using SMAPE metric)

In [38]:
!pip install -Uq upgini catboost


**INITIALIZATION**

In [39]:
from os.path import exists
import pandas as pd

df_path="train.csv_zip" if exists("train.csv.zip") else ("https://github.com/upgini/upgini/raw/main/notebooks/train.csv.zip")
df=pd.read_csv(df_path)
df.head()


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


**DATA CLEANING and SORTING**

In [40]:
df=df.sample(n=19000)
df["store"]= df["store"].astype(str)
df["item"] = df["item"].astype(str)
df["date"] = pd.to_datetime(df["date"])

df.sort_values ("date", inplace=True)
df.reset_index(inplace=True, drop=True)
df.head()


,date,store,item,sales
0,2013-01-01,1,41,6
1,2013-01-01,1,20,22
2,2013-01-01,10,38,47
3,2013-01-01,6,41,13
4,2013-01-01,9,29,34


**Training and Testing Sets + ENRICH FEATURES**

In [ ]:
from upgini import FeaturesEnricher, SearchKey
from upgini.metadata import CVType

train= df[df["date"] < "2017-01-01"]
test= df[df["date"] >= "2017-01-01"]


train_features= train.drop(columns=["sales"])
train_target= train["sales"]
test_features= test.drop(columns=["sales"])
test_target= test["sales"]


enricher = FeaturesEnricher (

  search_keys= {"date": SearchKey.DATE},
  cv= CVType.time_series
)
enricher.fit(train_features,
             train_target,
             eval_set=[(test_features , test_target)])

[=============================================               ] 76% Searching relevant features...

WARNING #1: Search started with DATE search key only
Try to add other keys like the COUNTRY, POSTAL_CODE, PHONE NUMBER, EMAIL/HEM, IP to your training dataset
for search through all the available data sources.
See docs https://github.com/upgini/upgini#-total-239-countries-and-up-to-41-years-of-history

Detected task type: ModelTaskType.REGRESSION. Reason: date search key is present, treating as regression
You can set task type manually with argument `model_task_type` of FeaturesEnricher constructor if task type detected incorrectly

WARNING #2: Your training sample is unstable in number of rows per date. It is recommended to redesign the training sample



Column name,Status,Errors
date,All valid,-
target,All valid,-




Running search request, search_id=53f6792f-f4a4-4165-8787-3319be6d9f89
We'll send email notification once it's completed, just use your personal api_key from profile.upgini.com


In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import eval_metric

model = CatBoostRegressor(verbose=False, allow_writing_files=False, random_state=0)

enriched.calculate_metrics (train_features, train_target, eval_set=[(test_features, test_target)], estimator=model, scoring="mean_absolute_percentage_error")


In [ ]:
enriched_train_features = enricher.transform(train_features, keep_input=True)
enriched_test_features = enricher.transform(test_features, keep_input=True)
enriched_train_features.head()


**Check Output:simple**

In [ ]:
model.fit(train_features, train_target)
preds= model.predict(test_features)
eval_metric(test_target.values, preds, "SMAPE")

**Check modified output**

In [ ]:
model.fit(enriched_train_features, train_target)
enriched_preds= model.predict(enriched_test_features)
eval_metric(test_target.values, enriched_preds, "SMAPE")